Creates the `cities` table (cities with 500+ active listings joined to Numbeo price-per-m²) and the `city_metrics` table (median price, nights booked, occupancy, revenue, and ROI for city centre vs. outside, with prices trimmed for outliers).

In [1]:
import duckdb

# Build the cities table: cities with 500+ active entire homes, joined to Numbeo property prices
duckdb.sql("""
CREATE OR REPLACE TABLE cities AS
WITH my_cities AS (
    SELECT parse_filename(filename, true) AS city,
           COUNT(*) AS active_entire_homes
    FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
    WHERE parse_filename(filename, true) NOT IN ('geneva','zurich','vaud')
      AND number_of_reviews_ltm > 0
      AND room_type = 'Entire home/apt'
    GROUP BY parse_filename(filename, true)
    HAVING COUNT(*) >= 500
),
numbeo AS (
    SELECT city AS numbeo_city,
           replace(lower(split_part(city, ',', 1)), ' ', '-') AS join_key,
           price_sqm
    FROM read_csv_auto('../data/reference/numbeo_centre.csv')
),
city_lookup AS (
    SELECT * FROM read_csv_auto('../data/reference/city_lookup.csv')
)
SELECT m.city, m.active_entire_homes, n.numbeo_city, n.price_sqm
FROM my_cities m
LEFT JOIN city_lookup c ON m.city = c.city
LEFT JOIN numbeo n ON n.numbeo_city = c.numbeo_city
                   OR (c.numbeo_city IS NULL AND n.join_key = m.city)
""")

duckdb.sql("DELETE FROM cities WHERE price_sqm IS NULL")

# Build city_metrics: the final results table, 80 cities, prices trimmed at the 1st and 99th percentile
duckdb.sql("""
CREATE OR REPLACE TABLE city_metrics AS
WITH raw AS (
    SELECT
        replace(parse_filename(filename, true), '.csv', '') AS city,
        CAST(replace(replace(price, '$', ''), ',', '') AS DOUBLE) AS price_num,
        number_of_reviews_ltm,
        minimum_nights
    FROM read_csv_auto('../data/detailed/*.csv.gz', filename = true, union_by_name = true)
    WHERE room_type = 'Entire home/apt'
      AND number_of_reviews_ltm > 0
      AND bedrooms <= 2
      AND price IS NOT NULL
),
bounds AS (
    SELECT city,
           quantile_cont(price_num, 0.01) AS lo,
           quantile_cont(price_num, 0.99) AS hi
    FROM raw
    GROUP BY city
),
listing_nights AS (
    SELECT
        r.city,
        r.price_num,
        LEAST((r.number_of_reviews_ltm / 0.5) * GREATEST(3, r.minimum_nights), 255) AS nights_booked,
        (r.number_of_reviews_ltm / 0.5) * GREATEST(3, r.minimum_nights) AS nights_uncapped
    FROM raw r
    JOIN bounds b ON b.city = r.city
    WHERE r.price_num BETWEEN b.lo AND b.hi
),
agg AS (
    SELECT
        city,
        MEDIAN(price_num) AS med_price_local,
        MEDIAN(nights_booked) AS med_nights,
        ROUND(100.0 * SUM(CASE WHEN nights_uncapped > 255 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_capped,
        COUNT(*) AS n_listings
    FROM listing_nights
    GROUP BY 1
    HAVING COUNT(*) >= 500
),
cur AS (SELECT * FROM read_csv_auto('../data/reference/country_currency.csv')),
ex  AS (SELECT * FROM read_csv_auto('../data/reference/exchange_rates.csv')),
out AS (SELECT city AS numbeo_city, price_sqm AS price_sqm_out
        FROM read_csv_auto('../data/reference/numbeo_outside.csv')),
utl AS (SELECT city AS numbeo_city, price_utilities
        FROM read_csv_auto('../data/reference/numbeo_utilities.csv'))
SELECT
    split_part(c.numbeo_city, ',', 1) AS city,
    trim(split_part(c.numbeo_city, ',', -1)) AS country,
    c.city AS city_slug,
    a.n_listings,
    ROUND(a.med_price_local / ex.usd_rate, 2) AS med_airbnb_price_usd,
    ROUND(a.med_nights, 0) AS med_nights,
    ROUND(100.0 * a.med_nights / 365, 1) AS occupancy_pct,
    ROUND(a.med_price_local / ex.usd_rate * a.med_nights, 0) AS annual_revenue_usd,
    a.pct_capped,
    c.price_sqm AS price_sqm_centre,
    out.price_sqm_out,
    utl.price_utilities,
    ROUND(100.0 * (
          (a.med_price_local / ex.usd_rate * a.med_nights)
        - (12 * utl.price_utilities * 60 / 85)
        - (0.01 * c.price_sqm * 60)
      ) / (c.price_sqm * 60), 2) AS roi_centre,
    ROUND(100.0 * (
          (a.med_price_local / ex.usd_rate * a.med_nights)
        - (12 * utl.price_utilities * 60 / 85)
        - (0.01 * out.price_sqm_out * 60)
      ) / (out.price_sqm_out * 60), 2) AS roi_outside
FROM cities c
JOIN agg a   ON a.city = c.city
JOIN cur     ON cur.country = trim(split_part(c.numbeo_city, ',', -1))
JOIN ex      ON ex.currency = cur.currency
LEFT JOIN out ON out.numbeo_city = c.numbeo_city
LEFT JOIN utl ON utl.numbeo_city = c.numbeo_city
WHERE c.city <> 'new-york-city'
""")

Counts listings per neighbourhood in the raw Los Angeles CSV to inspect neighbourhood granularity.

In [2]:
duckdb.sql("""
SELECT neighbourhood, COUNT(*) AS n
FROM read_csv_auto('../data/los-angeles.csv')
GROUP BY neighbourhood ORDER BY n DESC LIMIT 50
""").df()

,neighbourhood,n
0,Long Beach,1856
1,Hollywood,1598
2,Venice,1544
3,West Hollywood,1282
4,Santa Monica,1229
5,Downtown,1090
6,Exposition Park,906
7,Pasadena,805
8,Beverly Hills,791
9,Glendale,722


Finds Numbeo city-centre price rows whose normalized join key collides with more than one city name.

In [3]:
duckdb.sql("""
SELECT replace(lower(split_part(city, ',', 1)), ' ', '-') AS join_key,
       COUNT(*) AS n, string_agg(city, ' | ') AS matches
FROM read_csv_auto('../data/reference/numbeo_centre.csv')
GROUP BY 1 HAVING COUNT(*) > 1
""").df()

,join_key,n,matches
0,london,2,"London, United Kingdom | London, Canada"
1,birmingham,2,"Birmingham, United Kingdom | Birmingham, AL, U..."
2,vancouver,2,"Vancouver, Canada | Vancouver, WA, United States"
3,san-jose,2,"San Jose, CA, United States | San Jose, Costa ..."


Counts how many city URLs fall under each snapshot date extracted from the URL list.

In [4]:
duckdb.sql("""
SELECT
    regexp_extract(column0, '(\\d{4}-\\d{2}-\\d{2})', 1) AS snapshot_date,
    COUNT(*) AS n
FROM read_csv('../data/reference/city_urls.txt', header = false, columns = {'column0': 'VARCHAR'})
GROUP BY 1
ORDER BY 1
""").df()

,snapshot_date,n
0,2026-06-14,4
1,2026-06-15,15
2,2026-06-16,5
3,2026-06-19,5
4,2026-06-20,3
5,2026-06-21,5
6,2026-06-22,5
7,2026-06-23,5
8,2026-06-24,6
9,2026-06-25,3


Fetches current USD exchange rates from the Frankfurter API, adds manual overrides for unsupported currencies, and saves them to a CSV.

In [5]:
import requests, pandas as pd

api = requests.get("https://api.frankfurter.dev/v1/2026-06-30?base=USD",
                   headers={"User-Agent": "Mozilla/5.0"}).json()["rates"]

manual = {"COP": 3443.59, "ARS": 1450.00, "CLP": 922.34, "KES": 129.41, "TWD": 31.85}

rates = {**api, **manual, "USD": 1.0}

pd.DataFrame(rates.items(), columns=["currency", "usd_rate"]) \
  .to_csv("../data/reference/exchange_rates.csv", index=False)

Computes the median minimum-nights requirement per city across all raw listing files, sorted descending.

In [6]:
duckdb.sql("""
SELECT
    parse_filename(filename, true) AS city,
    MEDIAN(minimum_nights) AS med_min_nights
FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
WHERE room_type = 'Entire home/apt'
  AND number_of_reviews_ltm > 0
GROUP BY parse_filename(filename, true)
ORDER BY med_min_nights DESC
LIMIT 50
""").df()

,city,med_min_nights
0,new-york-city,30.0
1,singapore,6.0
2,ottawa,3.0
3,san-francisco,3.0
4,sunshine-coast,2.0
5,belize,2.0
6,cape-town,2.0
7,hong-kong,2.0
8,ireland,2.0
9,vienna,2.0


Counts listings per neighbourhood in the raw Clark County (Las Vegas) CSV.

In [7]:
duckdb.sql("""
SELECT neighbourhood, COUNT(*) AS n
FROM read_csv_auto('../data/clark-county-nv.csv')
GROUP BY 1 ORDER BY 2 DESC LIMIT 10
""").df()

,neighbourhood,n
0,Unincorporated Areas,15821
1,City of Las Vegas,2498
2,City of Henderson,897
3,City of North Las Vegas,868
4,City of Mesquite,187
5,Boulder City,19
6,Nellis AFB,6


Builds the list of detailed-listings download URLs, keeping only the ones whose city matches a city already in the `cities` table.

In [8]:
import unicodedata

def clean(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()

urls = open('../data/reference/city_urls.txt', encoding="utf-8").read().splitlines()
keep = duckdb.sql("SELECT city FROM cities").df()["city"].tolist()

out = []
for u in urls:
    name = clean(u.split("/")[-4])
    if name in keep:
        out.append(u.replace("visualisations/listings.csv", "data/listings.csv.gz"))

open('../data/reference/detailed_urls.txt', 'w', encoding="utf-8").write("\n".join(out))
print(len(out))

84


Downloads each detailed listings CSV (gzip) from Inside Airbnb into the local `data/detailed` folder.

In [9]:
import requests, unicodedata, pathlib
from urllib.parse import quote

pathlib.Path('../data/detailed').mkdir(exist_ok=True)

for i, u in enumerate(out, 1):
    name = clean(u.split("/")[-4])
    dest = pathlib.Path(f'../data/detailed/{name}.csv.gz')
    if dest.exists():
        continue
    r = requests.get(quote(u, safe=":/"), headers={"User-Agent": "Mozilla/5.0"})
    if r.status_code == 200:
        dest.write_bytes(r.content)
        print(i, name, len(r.content) // 1024, "KB")
    else:
        print(i, name, "FAILED", r.status_code)

Reports how many detailed city files were downloaded and lists which target cities are still missing.

In [10]:
import pathlib
got = [p.stem.replace('.csv','') for p in pathlib.Path('../data/detailed').glob('*.csv.gz')]
print(len(got), "downloaded")
print("missing:", [c for c in keep if c not in got])

84 downloaded
missing: []


Computes the percentage of listings with missing bedroom counts per city, from the detailed data.

In [11]:
duckdb.sql(""" 
SELECT parse_filename(filename, true) AS city,
COUNT(*) AS n,
ROUND(100.0*SUM(CASE WHEN bedrooms IS NULL THEN 1 ELSE 0 END)/ COUNT(*),1 ) AS pct_missing_bedrooms
FROM read_csv_auto('../data/detailed/*.csv.gz', filename = true, union_by_name = true)
WHERE room_type = 'Entire home/apt'
AND number_of_reviews_ltm >0
GROUP BY 1
ORDER BY pct_missing_bedrooms DESC
LIMIT 50
""").df()

,city,n,pct_missing_bedrooms
0,new-york-city.csv,5763,18.5
1,boston.csv,1943,16.7
2,buenos-aires.csv,20895,14.8
3,hong-kong.csv,1297,14.6
4,riga.csv,2520,14.4
5,lyon.csv,4100,13.9
6,porto.csv,10360,12.7
7,oakland.csv,1166,12.6
8,paris.csv,37404,12.5
9,portland.csv,2629,12.1


Compares median nights-booked and reviews for London listings with all bedroom counts versus only 1-2 bedroom listings.

In [12]:
duckdb.sql("""
SELECT
    'detailed, all bedrooms' AS variant,
    MEDIAN(LEAST((number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights), 255)) AS med_nights,
    MEDIAN(minimum_nights) AS med_min_nights,
    MEDIAN(number_of_reviews_ltm) AS med_reviews,
    COUNT(*) AS n
FROM read_csv_auto('../data/detailed/london.csv.gz')
WHERE room_type = 'Entire home/apt' AND number_of_reviews_ltm > 0
UNION ALL
SELECT
    'detailed, 1-2 bed',
    MEDIAN(LEAST((number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights), 255)),
    MEDIAN(minimum_nights),
    MEDIAN(number_of_reviews_ltm),
    COUNT(*)
FROM read_csv_auto('../data/detailed/london.csv.gz')
WHERE room_type = 'Entire home/apt' AND number_of_reviews_ltm > 0 AND bedrooms <= 2
""").df()

,variant,med_nights,med_min_nights,med_reviews,n
0,"detailed, all bedrooms",42.0,2.0,6.0,33455
1,"detailed, 1-2 bed",42.0,2.0,6.0,25785


Recomputes ROI for every city assuming 85% occupancy instead of 60%, alongside the original ROI.

In [13]:
duckdb.sql("""
SELECT city, country, roi_centre AS roi_at_60,
    ROUND(100.0 * (med_airbnb_price_usd * med_nights) / (price_sqm_centre * 85)
          - 100.0 * (12 * price_utilities) / (85 * price_sqm_centre) - 1.0, 2) AS roi_at_85
FROM city_metrics
ORDER BY roi_centre DESC 
LIMIT 50
""").df()

,city,country,roi_at_60,roi_at_85
0,Chicago,United States,11.53,7.65
1,Edinburgh,United Kingdom,10.12,6.62
2,Portland,United States,9.77,6.30
3,Columbus,United States,9.74,6.31
4,Minneapolis,United States,8.22,5.32
5,Fort Worth,United States,8.12,5.22
6,Victoria,Canada,7.39,4.84
7,Barcelona,Spain,6.71,4.34
8,Winnipeg,Canada,6.61,4.07
9,San Diego,United States,6.45,4.14


Measures how much city rankings shift between the 60%-occupancy ROI and the 85%-occupancy ROI (max and average rank movement).

In [14]:
duckdb.sql("""
WITH r AS (
  SELECT city,
    RANK() OVER (ORDER BY roi_centre DESC) AS rank_60,
    RANK() OVER (ORDER BY
      100.0*(med_airbnb_price_usd*med_nights)/(price_sqm_centre*85)
      - 100.0*(12*price_utilities)/(85*price_sqm_centre) - 1.0 DESC) AS rank_85
  FROM city_metrics
)
SELECT MAX(ABS(rank_60 - rank_85)) AS biggest_move,
       ROUND(AVG(ABS(rank_60 - rank_85)), 2) AS avg_move
FROM r
""").df()

,biggest_move,avg_move
0,10,0.93


Lists the 50 cities whose ROI rank moves the most between the 60% and 85% occupancy assumptions.

In [15]:
duckdb.sql("""
WITH r AS (
  SELECT city, country, roi_centre,
    RANK() OVER (ORDER BY roi_centre DESC) AS rank_60,
    RANK() OVER (ORDER BY
      100.0*(med_airbnb_price_usd*med_nights)/(price_sqm_centre*85)
      - 100.0*(12*price_utilities)/(85*price_sqm_centre) - 1.0 DESC) AS rank_85
  FROM city_metrics
)
SELECT city, country, roi_centre, rank_60, rank_85, rank_85 - rank_60 AS move
FROM r ORDER BY ABS(rank_85 - rank_60) DESC LIMIT 50
""").df()

,city,country,roi_centre,rank_60,rank_85,move
0,Riga,Latvia,0.29,66,76,10
1,Bogota,Colombia,0.38,64,68,4
2,Thessaloniki,Greece,0.12,70,73,3
3,Manchester,United Kingdom,2.23,42,45,3
4,Sydney,Australia,0.96,62,59,-3
5,Buenos Aires,Argentina,1.13,58,61,3
6,Stockholm,Sweden,0.26,69,66,-3
7,Brussels,Belgium,4.09,23,25,2
8,Bangkok,Thailand,-0.12,74,72,-2
9,Tokyo,Japan,0.27,67,65,-2


Checks whether trimming price outliers at the 1st/99th percentile materially changes each city's median price.

In [16]:
# 3. Does outlier trimming change the medians? The README claims it doesn't.
duckdb.sql("""
WITH p AS (
  SELECT replace(parse_filename(filename,true),'.csv','') AS city,
         CAST(replace(replace(price,'$',''),',','') AS DOUBLE) AS pr
  FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
  WHERE room_type='Entire home/apt' AND number_of_reviews_ltm>0 AND bedrooms<=2
),
b AS (
  SELECT city,
         quantile_cont(pr, 0.01) AS lo,
         quantile_cont(pr, 0.99) AS hi
  FROM p GROUP BY city
)
SELECT p.city,
       ROUND(MEDIAN(p.pr), 2) AS med_all,
       ROUND(MEDIAN(CASE WHEN p.pr BETWEEN b.lo AND b.hi THEN p.pr END), 2) AS med_trimmed
FROM p JOIN b ON b.city = p.city
GROUP BY p.city
ORDER BY ABS(med_all - med_trimmed) DESC
LIMIT 10
""").df()

,city,med_all,med_trimmed
0,bogota,174553.00,174500.00
1,nairobi,5985.00,5965.67
2,budapest,30877.75,30872.50
3,santiago,62463.00,62462.00
4,new-york-city,200.97,200.07
5,athens,99.00,99.01
6,bordeaux,110.50,110.50
7,thessaloniki,74.00,74.00
8,quebec-city,234.00,234.00
9,dublin,228.25,228.25


Counts how many cities in the `cities` table have at least 500 qualifying 1-2 bedroom detailed listings (excluding New York City).

In [17]:
duckdb.sql("""
SELECT COUNT(*) FROM cities c
JOIN (SELECT replace(parse_filename(filename,true),'.csv','') AS city, COUNT(*) n
      FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
      WHERE room_type='Entire home/apt' AND number_of_reviews_ltm>0 AND bedrooms<=2
      GROUP BY 1 HAVING COUNT(*)>=500) a ON a.city=c.city
WHERE c.city <> 'new-york-city'
""").df()

,count_star()
0,82


Compares how many cities would qualify under a strict bedroom filter versus a fallback filter that also allows listings with a missing bedroom count.

In [18]:
duckdb.sql("""
SELECT
    COUNT(*) AS cities_strict,
    SUM(CASE WHEN n_with_fallback >= 500 AND n_strict < 500 THEN 1 ELSE 0 END) AS would_be_added
FROM (
    SELECT replace(parse_filename(filename,true),'.csv','') AS city,
        SUM(CASE WHEN bedrooms <= 2 THEN 1 ELSE 0 END) AS n_strict,
        SUM(CASE WHEN bedrooms <= 2 OR (bedrooms IS NULL AND accommodates <= 4) THEN 1 ELSE 0 END) AS n_with_fallback
    FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
    WHERE room_type='Entire home/apt' AND number_of_reviews_ltm>0
    GROUP BY 1
)
""").df()

,cities_strict,would_be_added
0,84,0.0


Finds cities where the maximum listing price is far above the 99th percentile price, to spot remaining outliers.

In [19]:
duckdb.sql("""
SELECT replace(parse_filename(filename,true),'.csv','') AS city,
    ROUND(MEDIAN(pr),0) AS median,
    ROUND(quantile_cont(pr,0.99),0) AS p99,
    ROUND(MAX(pr),0) AS max_price,
    ROUND(MAX(pr)/quantile_cont(pr,0.99),1) AS max_over_p99
FROM (SELECT filename, CAST(replace(replace(price,'$',''),',','') AS DOUBLE) AS pr
      FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
      WHERE room_type='Entire home/apt' AND number_of_reviews_ltm>0 AND bedrooms<=2)
GROUP BY 1 ORDER BY max_over_p99 DESC LIMIT 50
""").df()

,city,median,p99,max_price,max_over_p99
0,buenos-aires,103226.0,443003.0,171643781.0,387.5
1,bangkok,1619.0,6680.0,1157979.0,173.3
2,bogota,174553.0,1083502.0,174600081.0,161.1
3,prague,2519.0,8345.0,1191906.0,142.8
4,santiago,62463.0,445796.0,43741160.0,98.1
5,san-diego,303.0,1089.0,82341.0,75.6
6,sao-paulo,339.0,1107.0,57127.0,51.6
7,budapest,30878.0,160315.0,8258389.0,51.5
8,rio-de-janeiro,422.0,2240.0,114118.0,50.9
9,milan,137.0,675.0,28587.0,42.4


Exports the final `city_metrics` table to `results.csv` and previews the top 12 cities by ROI.

In [20]:
# 1. export + top of ranking
d = duckdb.sql("SELECT * FROM city_metrics ORDER BY roi_centre DESC").df()
d.to_csv('../data/reference/results.csv', index=False)
print(len(d))
d[["city","country","roi_centre","roi_outside","occupancy_pct","annual_revenue_usd"]].head(12)

80


,city,country,roi_centre,roi_outside,occupancy_pct,annual_revenue_usd
0,Chicago,United States,11.53,18.03,39.5,32426.0
1,Edinburgh,United Kingdom,10.12,14.89,39.5,47787.0
2,Portland,United States,9.77,11.45,42.7,27648.0
3,Columbus,United States,9.74,19.80,39.5,24048.0
4,Minneapolis,United States,8.22,13.40,27.9,20132.0
5,Fort Worth,United States,8.12,19.52,34.5,24255.0
6,Victoria,Canada,7.39,9.11,41.1,28137.0
7,Barcelona,Spain,6.71,10.22,41.1,36817.0
8,Winnipeg,Canada,6.61,6.60,34.5,12126.0
9,San Diego,United States,6.45,8.17,34.5,38225.0


Lists cities with negative city-centre ROI, plus Boston for reference.

In [21]:
# 2. negative cities + Boston
duckdb.sql("""
SELECT city, country, roi_centre, roi_outside, occupancy_pct, annual_revenue_usd
FROM city_metrics WHERE roi_centre < 0 OR city = 'Boston' ORDER BY roi_centre
""").df()

,city,country,roi_centre,roi_outside,occupancy_pct,annual_revenue_usd
0,Oslo,Norway,-0.74,-0.56,8.2,5434.0
1,Hong Kong,Hong Kong (China),-0.68,-0.48,15.3,7370.0
2,Munich,Germany,-0.49,-0.26,9.9,7260.0
3,Nairobi,Kenya,-0.27,0.17,6.6,1106.0
4,London,United Kingdom,-0.26,0.75,11.5,12161.0
5,Copenhagen,Denmark,-0.16,0.17,8.2,7489.0
6,Bangkok,Thailand,-0.12,0.63,23.0,4094.0
7,Vienna,Austria,-0.07,1.22,21.4,10986.0
8,Boston,United States,3.93,9.44,36.2,42680.0


Prints correlations between revenue, price, and ROI, and shows the average centre-vs-outside ROI spread by country.

In [22]:
# 3. correlations + spread by country
print(d["annual_revenue_usd"].corr(d["roi_centre"]),
      d["price_sqm_centre"].corr(d["roi_centre"]),
      d["annual_revenue_usd"].corr(d["price_sqm_centre"]))
d["spread"] = d["roi_outside"] - d["roi_centre"]
g = d.groupby("country").agg(n=("city","count"), mean_spread=("spread","mean"))
print(g[g["n"] >= 3].sort_values("mean_spread", ascending=False).round(2))

0.7715156619641029 -0.36116345612902995 0.0735934163250448
                 n  mean_spread
country                        
United States   21         5.10
Spain            5         3.75
Belgium          3         2.73
Italy            6         2.31
Canada           7         1.83
United Kingdom   4         1.71
Australia        3         1.03
France           3         0.90


Lists cities present in the `cities` table but dropped from the final `city_metrics` table.

In [23]:
duckdb.sql("SELECT city FROM cities WHERE city NOT IN (SELECT city_slug FROM city_metrics) AND city <> 'new-york-city'").df()

,city
0,rotterdam
1,the-hague
2,rochester


Prints correlations between capped-nights rate, occupancy, nightly price, listing count, and ROI.

In [24]:
print("capped vs occupancy:", d["pct_capped"].corr(d["occupancy_pct"]))
print("nightly vs property price:", d["med_airbnb_price_usd"].corr(d["price_sqm_centre"]))
print("listings vs roi:", d["n_listings"].corr(d["roi_centre"]))

capped vs occupancy: 0.8774255336225322
nightly vs property price: 0.4090684963426397
listings vs roi: -0.3717192924606052


Estimates what share of ROI's variation is attributable to revenue versus property price, using log-variance decomposition.

In [25]:
import numpy as np
lr = np.log(d["annual_revenue_usd"]); lm = np.log(d["price_sqm_centre"])
print("var log revenue:", round(lr.var(),4))
print("var log price:  ", round(lm.var(),4))
print("covariance:     ", round(np.cov(lr,lm)[0,1],4))
print("revenue share of ROI variation:", round(lr.var()/(lr.var()+lm.var()),3))

var log revenue: 0.4607
var log price:   0.2857
covariance:      0.13
revenue share of ROI variation: 0.617


Prints correlations between nightly price, utility costs, and property price.

In [26]:
print("nightly vs utilities: ", round(d["med_airbnb_price_usd"].corr(d["price_utilities"]),3))
print("property vs utilities:", round(d["price_sqm_centre"].corr(d["price_utilities"]),3))

nightly vs utilities:  0.437
property vs utilities: 0.385


Compares the raw versus log-scale correlation between annual revenue and property price.

In [27]:
print("raw:", round(d["annual_revenue_usd"].corr(d["price_sqm_centre"]),3))
print("log:", round(np.log(d["annual_revenue_usd"]).corr(np.log(d["price_sqm_centre"])),3))

raw: 0.074
log: 0.358


Prints the max/min ratio for several metrics, the peak occupancy and capped-nights rate, and a slice of cities ranked by ROI around the middle of the distribution.

In [28]:
# 1. spreads
for c in ["med_airbnb_price_usd","occupancy_pct","price_sqm_centre"]:
    print(c, round(d[c].max()/d[c].min(),1))

# 2. does any city's median approach the cap?
print("max occupancy:", d["occupancy_pct"].max(), "| max pct_capped:", d["pct_capped"].max())

# 3. the ROI crowding around Riga
print(d.sort_values("roi_centre", ascending=False)[["city","roi_centre"]].iloc[60:78].to_string())

med_airbnb_price_usd 7.8
occupancy_pct 7.1
price_sqm_centre 19.6
max occupancy: 46.8 | max pct_capped: 29.1
              city  roi_centre
60  Rio de Janeiro        0.97
61          Sydney        0.96
62        Bordeaux        0.62
63          Bogota        0.38
64           Paris        0.33
65            Riga        0.29
66           Tokyo        0.27
67       Amsterdam        0.27
68       Stockholm        0.26
69    Thessaloniki        0.12
70          Taipei        0.11
71           Milan        0.09
72          Vienna       -0.07
73         Bangkok       -0.12
74      Copenhagen       -0.16
75          London       -0.26
76         Nairobi       -0.27
77          Munich       -0.49


For cities with some null-priced listings, compares average availability and review counts between priced and unpriced listings.

In [29]:
duckdb.sql("""
SELECT
    replace(parse_filename(filename,true),'.csv','') AS city,
    ROUND(AVG(CASE WHEN price IS NOT NULL THEN availability_365 END),0) AS avail_priced,
    ROUND(AVG(CASE WHEN price IS NULL     THEN availability_365 END),0) AS avail_null,
    ROUND(AVG(CASE WHEN price IS NOT NULL THEN number_of_reviews_ltm END),1) AS rev_priced,
    ROUND(AVG(CASE WHEN price IS NULL     THEN number_of_reviews_ltm END),1) AS rev_null,
    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS n_null
FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
WHERE room_type='Entire home/apt' AND number_of_reviews_ltm > 0 AND bedrooms <= 2
GROUP BY 1
HAVING SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) > 0
ORDER BY rev_null DESC
LIMIT 50
""").df()

,city,avail_priced,avail_null,rev_priced,rev_null,n_null
0,quebec-city,229.0,172.0,26.2,27.1,15.0
1,porto,259.0,232.0,19.5,20.5,535.0
2,athens,257.0,250.0,18.8,19.2,20.0
3,edinburgh,148.0,44.0,30.1,17.6,268.0
4,bogota,305.0,290.0,15.9,16.9,45.0
5,antwerp,215.0,153.0,20.8,16.2,133.0
6,winnipeg,260.0,59.0,24.0,16.0,23.0
7,budapest,184.0,30.0,26.4,14.7,462.0
8,mexico-city,255.0,50.0,23.4,14.6,308.0
9,brussels,202.0,38.0,25.8,14.3,250.0


Computes each city's utility cost as a share of ROI base and prints the 10 cities where utilities matter most.

In [30]:
d["util_share"] = (12*d["price_utilities"])/(85*d["price_sqm_centre"])*100
print(d.nlargest(10,"util_share")[["city","roi_centre","util_share"]].to_string())

            city  roi_centre  util_share
65          Riga        0.29    1.659715
57  Buenos Aires        1.13    1.103035
2       Portland        9.77    1.052338
8       Winnipeg        6.61    1.019931
3       Columbus        9.74    0.914815
41    Manchester        2.23    0.853687
69  Thessaloniki        0.12    0.843911
59       Bergamo        1.01    0.821002
51        Athens        1.55    0.797636
52      Santiago        1.54    0.789110


In [32]:
duckdb.sql("SELECT * FROM city_metrics ORDER BY roi_centre DESC").df().to_csv('../data/reference/results.csv', index=False)